In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np


df = pd.read_csv('credit.csv')



print("--- Task 1 & 2: K-Means Clustering on Numeric Columns ---")


numeric_cols = ['months_loan_duration', 'amount', 'percent_of_income', 
                'years_at_residence', 'age', 'existing_loans_count', 
                'dependents']
df_numeric = df[numeric_cols]


scaler = StandardScaler()
df_numeric_scaled = scaler.fit_transform(df_numeric)


results_task1 = []
for k in [2, 3, 4]:
  
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_numeric_scaled)
    inertia = kmeans.inertia_
    silhouette_avg = silhouette_score(df_numeric_scaled, kmeans.labels_)
    results_task1.append({'K': k, 'Inertia': inertia, 'Silhouette Score': silhouette_avg, 'Labels': kmeans.labels_})


print("\nTask 1 Results (K-Means on Numeric Columns):")
task1_results_df = pd.DataFrame([{'K': res['K'], 'Inertia': res['Inertia'], 'Silhouette Score': res['Silhouette Score']} for res in results_task1])
print(task1_results_df.to_markdown(index=False, numalign="left", stralign="left"))






--- Task 1 & 2: K-Means Clustering on Numeric Columns ---

Task 1 Results (K-Means on Numeric Columns):
| K   | Inertia   | Silhouette Score   |
|:----|:----------|:-------------------|
| 2   | 5865.37   | 0.23057            |
| 3   | 4994.96   | 0.254237           |
| 4   | 4307.75   | 0.202753           |


In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


df = pd.read_csv('credit.csv')


numeric_cols = ['months_loan_duration', 'amount', 'percent_of_income', 
                'years_at_residence', 'age', 'existing_loans_count', 
                'dependents']
df_numeric = df[numeric_cols]

scaler = StandardScaler()
df_numeric_scaled = scaler.fit_transform(df_numeric)

kmeans_k2 = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_k2.fit(df_numeric_scaled)
k2_labels = kmeans_k2.labels_




print("--- Task 2: Cluster Profiling (K=2) ---")

df_numeric_for_profile = df_numeric.copy()
df_numeric_for_profile['Cluster'] = k2_labels

cluster_profile_task2 = df_numeric_for_profile.groupby('Cluster')[numeric_cols].mean()

cluster_size_task2 = df_numeric_for_profile['Cluster'].value_counts().rename('Size')
cluster_profile_task2 = pd.concat([cluster_profile_task2, cluster_size_task2], axis=1)
cluster_profile_task2['Size_Percent'] = (cluster_profile_task2['Size'] / cluster_profile_task2['Size'].sum()) * 100

print("Cluster Profile (K=2, Numeric Columns):")
print(cluster_profile_task2.to_markdown(numalign="left", stralign="left"))

--- Task 2: Cluster Profiling (K=2) ---
Cluster Profile (K=2, Numeric Columns):
| Cluster   | months_loan_duration   | amount   | percent_of_income   | years_at_residence   | age     | existing_loans_count   | dependents   | Size   | Size_Percent   |
|:----------|:-----------------------|:---------|:--------------------|:---------------------|:--------|:-----------------------|:-------------|:-------|:---------------|
| 0         | 37.0092                | 7416.16  | 2.62844             | 2.89908              | 35.9862 | 1.48624                | 1.18807      | 218    | 21.8           |
| 1         | 16.413                 | 2115.77  | 3.06905             | 2.82992              | 35.4233 | 1.38491                | 1.14578      | 782    | 78.2           |


In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('credit.csv')



print("--- Task 3: K-Means Clustering on OHE Data ---")


df_task3 = df.drop(columns=['default'])


df_task3_ohe = pd.get_dummies(df_task3, drop_first=False) 


scaler_task3 = StandardScaler()
df_task3_scaled = scaler_task3.fit_transform(df_task3_ohe)


results_task3 = []
for k in [2, 3, 4]:
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_task3_scaled)
    inertia = kmeans.inertia_
    
 
    silhouette_avg = silhouette_score(df_task3_scaled, kmeans.labels_)
    results_task3.append({'K': k, 'Inertia': inertia, 'Silhouette Score': silhouette_avg})


print("\nTask 3 Results (K-Means on OHE Data):")
task3_results_df = pd.DataFrame(results_task3)
print(task3_results_df.to_markdown(index=False, numalign="left", stralign="left"))

best_k_task3 = task3_results_df.loc[task3_results_df['Silhouette Score'].idxmax()]['K']
print(f"\nBest K for Task 3 is K={int(best_k_task3)} based on the highest Silhouette Score.")

--- Task 3: K-Means Clustering on OHE Data ---

Task 3 Results (K-Means on OHE Data):
| K   | Inertia   | Silhouette Score   |
|:----|:----------|:-------------------|
| 2   | 41508     | 0.0661314          |
| 3   | 39931.9   | 0.0531377          |
| 4   | 38773.3   | 0.0484761          |

Best K for Task 3 is K=2 based on the highest Silhouette Score.


In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist


df = pd.read_csv('credit.csv')




df_task4_ohe = df.drop(columns=['default'])
df_task4_ohe = pd.get_dummies(df_task4_ohe, drop_first=False) 


scaler = StandardScaler()
df_task4_scaled = scaler.fit_transform(df_task4_ohe)


euclidean_dist_condensed = pdist(df_task4_scaled, metric='euclidean')


linkages = ["ward", "single", "average"]
results_task4_alt = []

for link_method in linkages:
    
    Z = linkage(euclidean_dist_condensed, method=link_method)

    
    for k in [2, 3, 4]:
        
        cluster_labels = fcluster(Z, k, criterion='maxclust')
        
        
        silhouette_avg = silhouette_score(df_task4_scaled, cluster_labels, metric='euclidean')
        results_task4_alt.append({'Linkage': link_method, 'K': k, 'Silhouette Score': silhouette_avg})


task4_alt_results_df = pd.DataFrame(results_task4_alt)
